## Example Notebook: Using ImageAgent

### 🛠️ Setup Instructions

Before running this notebook:
- Make sure all required libraries are installed by running:

    ```bash
    pip install ".[image-agent]"
    ```
### About
ImageAgent is built on top of the **Multi-Modal Critical Thinking (MMCT)** ([arxiv.org/abs/2405.18358](https://arxiv.org/abs/2405.18358)) architecture, which leverages two collaborative agents:

- **Planner**: Generates an initial response based on the provided input. It uses a set of default tools from `ImageQnaTools` but can be customized.
- **Critic (optional)**: Evaluates the planner’s response and provides feedback for improvement. This feedback loop helps increase accuracy and quality.

By default, the critic agent is enabled. Users can disable it by setting `use_critic_agent=False` during initialization.

> **Note:** Disabling the critic agent skips the feedback loop and may reduce the accuracy of the final response.

---

### Tool Configuration

The planner supports the following tools via the `ImageQnaTools` enum:

- `ImageQnaTools.object_detection` – This tool detects the object in the image.
- `ImageQnaTools.ocr` – for extracting text content.
- `ImageQnaTools.recog` – This tool recognise the objects in the image.
- `ImageQnaTools.vit` – for high-level visual understanding using vision llm.

Users can pass a list of tools via the `tools` parameter to override the defaults.

---


### Importing Libaries

In [1]:
# Import necessary modules

from mmct.providers.azure import AzureLLMProvider # Import Azure LLM Provider, You can create LLM providers for other vendor also using the BaseLLMProvider as described in next section of this notebook
from mmct.config.providers import ImageAgentProviderConfig
from azure.identity import DefaultAzureCredential, AzureCliCredential, ChainedTokenCredential
from mmct.image_pipeline import ImageAgent, ImageQnaTools
import nest_asyncio
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")
nest_asyncio.apply()

/home/v-amanpatkar/anaconda3/envs/migration_test/lib/python3.11/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
/home/v-amanpatkar/anaconda3/envs/migration_test/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/v-amanpatkar/anaconda3/envs/migration_test/lib/python3.11/site-packages/modelscope/utils/plugins.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### Configuring the ImageAgentProviderConfig

In [2]:
credentials = ChainedTokenCredential(AzureCliCredential(),DefaultAzureCredential())

In [3]:
provider = ImageAgentProviderConfig(
    llm_provider=AzureLLMProvider(
        endpoint = "https://geckooai.openai.azure.com/",#"<your_endpoint>",
        deployment_name="gpt-4o",#"<deployment_name>",
        model_name="gpt-4o", #"<model_name>",
        api_version="2024-08-01-preview",#"api_version",
        credentials=credentials,
    )
)


* You can use the api_key instead of credentials

### Executing ImageAgent

In [4]:
# Create ImageAgent instance
mmct_agent = ImageAgent(
    query="What objects are visible in the image?",#"user-query",
    image_path="/home/v-amanpatkar/work/demo/workdesk.jpg",#"image-path",
    tools=[ImageQnaTools.vit,ImageQnaTools.object_detection],
    use_critic_agent=True,
    stream=True,
    provider = provider
)

# Run the agent
response = await mmct_agent()
print("ImageAgent executed successfully!")
print(f"Response: {response}")

>>> AGENT START: planner

[USER]

query:What objects are visible in the image?, image path:/home/v-amanpatkar/work/demo/workdesk.jpg.
Always criticize the final response if planner asks for review and provide feedback.

>>> TOOL CALL: object_detect_tool

Arguments: {}

...

[TOOL]

[ASSISTANT]

Observation:

The ViT tool provided a comprehensive list of objects visible in the image:

1. Desk: A wooden desk with a smooth surface.
2. Office Chair: Ergonomic and adjustable, located at the desk.
3. Computer Monitor: Positioned on the desk, facing forward.
4. Keyboard: Centered on a long desk mat.
5. Mouse: Positioned next to the keyboard on the desk mat.
6. Desk Mat: A fabric mat covering part of the desk surface.
7. Laptop: Located underneath the monitor stand.
8. Monitor Stand: Wooden with a shelf for organizing items.
9. Desk Plant: A plant in a woven pot adding greenery to the workspace.
10. Microphone: Positioned next to the monitor, useful for audio recording.
11. Abstract Artwork: A framed piece on the wall displaying the text "BE FEARLESS."
12. Speakers: One speaker visible next to the monitor for audio output.
13. Shelving: White shelves above the desk for storage, partially visible.
14. Window: Partially visible on the left, providing natural light.

The Object Detection tool re-confirmed previous detections:
- Mouse
- Potted Plant
- Keyboard
- TV
- Chair

The additional tool use has identified more items including smaller and commonly missed objects such as the 
microphone, speakers, laptop, artwork, shelving, and desk mat.

{"Answer": "The image includes objects such as a desk, office chair, computer monitor, keyboard, mouse, desk mat, 
laptop, monitor stand, desk plant, microphone, abstract artwork with text \"BE FEARLESS\", speakers, shelving, and 
a window."} 

TERMINATE.

>>> AGENT END: planner

ImageAgent executed successfully!
Response: response='The objects visible in the image are a desk, office chair, computer monitor, keyboard, mouse, desk mat, laptop, monitor stand, desk plant, microphone, abstract artwork with the text "BE FEARLESS", speakers, shelving, and a window.' tokens=TokenInfo(input_token=4894, output_token=538)


In [ ]:
# Display the response
print(response)

## Example Implementation of LLMProvider from other vendor like Anthropic

In [ ]:
from mmct.providers.base import BaseLLMProvider
from typing import Dict, Any, List, Optional
import anthropic


class AnthropicLLMProvider(BaseLLMProvider):
    """Anthropic LLM provider implementation for Claude models."""

    def __init__(
        self,
        api_key: str,
        model_name: str = "claude-3-5-sonnet-20241022",
        timeout: Optional[int] = 600,
        max_retries: Optional[int] = 2,
    ):
        """Initialize AnthropicLLMProvider.

        Args:
            api_key: Anthropic API key for authentication
            model_name: Name of the Claude model (default: claude-3-5-sonnet-20241022)
            timeout: Request timeout in seconds (default: 600)
            max_retries: Maximum number of retry attempts (default: 2)

        Raises:
            ValueError: If required fields are missing
        """
        if not api_key:
            raise ValueError("Anthropic API key is required!")

        if not model_name:
            raise ValueError("Model name is required!")

        self.api_key = api_key
        self.model_name = model_name
        self.timeout = timeout
        self.max_retries = max_retries
        self.client = anthropic.AsyncAnthropic(
            api_key=self.api_key,
            timeout=self.timeout,
            max_retries=self.max_retries,
        )

    async def chat_completion(
        self, messages: List[Dict], **kwargs
    ) -> Dict[str, Any]:
        """Generate chat completion using Anthropic Claude API.

        Args:
            messages: List of message dictionaries with 'role' and 'content' keys
            **kwargs: Additional parameters like temperature, max_tokens, etc.

        Returns:
            Dict containing the response content, usage, model, and finish_reason
        """
        try:
            # Extract common parameters
            temperature = kwargs.get("temperature", 1.0)
            max_tokens = kwargs.get("max_tokens", 4096)
            top_p = kwargs.get("top_p", None)
            system = kwargs.get("system", None)

            # Convert OpenAI-style messages to Anthropic format
            # Anthropic separates system messages from the messages list
            anthropic_messages = []
            system_message = None

            for msg in messages:
                if msg.get("role") == "system":
                    system_message = msg.get("content")
                else:
                    anthropic_messages.append(
                        {"role": msg.get("role"), "content": msg.get("content")}
                    )

            # If system parameter is provided in kwargs, it takes precedence
            if system:
                system_message = system

            # Prepare API call parameters
            api_params = {
                "model": self.model_name,
                "messages": anthropic_messages,
                "max_tokens": max_tokens,
                "temperature": temperature,
            }

            # Add optional parameters
            if system_message:
                api_params["system"] = system_message

            if top_p is not None:
                api_params["top_p"] = top_p

            # Make the API call
            response = await self.client.messages.create(**api_params)

            # Format response to match the expected structure
            return {
                "content": response.content[0].text,
                "usage": {
                    "prompt_tokens": response.usage.input_tokens,
                    "completion_tokens": response.usage.output_tokens,
                    "total_tokens": response.usage.input_tokens
                    + response.usage.output_tokens,
                },
                "model": response.model,
                "finish_reason": response.stop_reason,
            }

        except Exception as e:
            raise Exception(f"Anthropic chat completion failed: {e}")

    def get_autogen_client(self, **kwargs):
        """Get autogen-compatible client for Anthropic.
        
        Args:
            **kwargs: Additional parameters like temperature
            
        Returns:
            Autogen-compatible Anthropic client
            
        Raises:
            Exception: If autogen_ext.models.anthropic is not available
        """
        try:
            # Try to import Anthropic client from autogen_ext
            from autogen_ext.models.anthropic import AnthropicChatCompletionClient
            
            temperature = kwargs.get("temperature", 1.0)
            max_tokens = kwargs.get("max_tokens", 4096)
            
            return AnthropicChatCompletionClient(
                model=self.model_name,
                api_key=self.api_key,
                temperature=temperature,
                max_tokens=max_tokens,
            )
        except ImportError:
            raise Exception(
                "autogen_ext.models.anthropic is not available. "
                "Please install autogen-ext with Anthropic support or use a different LLM provider. "
                "You can install it with: pip install 'autogen-ext[anthropic]'"
            )
        except Exception as e:
            raise Exception(f"Failed to create Anthropic autogen client: {e}")

    async def close(self):
        """Close the Anthropic client and cleanup resources."""
        if self.client:
            await self.client.close()


In [ ]:
# Example usage:
provider = ImageAgentProviderConfig(
    llm_provider=AnthropicLLMProvider(
        api_key="your-anthropic-api-key",
        model_name="claude-3-5-sonnet-20241022",
    )
)

# Create ImageAgent instance
mmct_agent = ImageAgent(
    query="What objects are visible in the image?",#"user-query",
    image_path="/home/v-amanpatkar/work/demo/workdesk.jpg",#"image-path",
    tools=[ImageQnaTools.vit,ImageQnaTools.object_detection],
    use_critic_agent=True,
    stream=True,
    provider = provider
)

# Run the agent
response = await mmct_agent()
print("ImageAgent executed successfully!")
print(f"Response: {response}")